In [33]:
import sys
from pathlib import Path

import numpy as np
import optuna
import pandas as pd
import torch
from astropy.io import fits
from sklearn.model_selection import KFold
from torch import nn
from torchvision import transforms
from torchvision.transforms import Normalize

In [34]:
cwd = Path.cwd()
home_dir = cwd.parent.parent
cutout_dir = home_dir / 'data' / 'cutouts'
utils_path = home_dir / 'utils.py'

In [35]:
sys.path.append(str(home_dir))
import utils as ut

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
OUT_FEATURES = ['FLAG']
SEED = 42
BEST_ACC = 0

# HYPERPARAMETERS
INPUT_SIZE = 67
EPOCHS = 200
NUM_TRIALS = 20
PATIENCE = 20
band_choice = 'norm'  # norm, sub, div
SCALER = 'tanh'
N_SPLITS = 5
AUG_MULT = 5
OPTIMIZE = True

In [37]:
if OPTIMIZE:
    COMMENT = input('Enter OPTIMIZATION comment: ')

if band_choice == 'norm':
    BANDS = ['f150w', 'f277w', 'f444w']
elif band_choice == 'div':
    BANDS = ['f150w/f444w', 'f277w/f444w']
elif band_choice == 'sub':
    BANDS = ['f444w-f150w', 'f444w-f277w']

In [38]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

if SEED:
    ut.set_seeds(SEED)

## Prepare Data

In [39]:
# LOAD DATA INTO DF

cols = {'idx': [], 'images': [], 'header': []}

for folder in cutout_dir.iterdir():
    if folder.is_dir():
        cols['idx'].append(folder.stem)

        temp_df = ut.fits_df(folder)
        temp_df.sort_values(by='BAND')

        # load images
        temp_imgs = {}
        for _, row in temp_df.iterrows():
            hdul = fits.open(row['FILEPATH'])
            data = hdul[0].data.astype('float32')
            header = hdul[0].header
            temp_imgs[row['BAND']] = data
            hdul.close()
        cols['images'].append(temp_imgs)
        cols['header'].append(header)

# turn dict into df and expand header columns
df = pd.DataFrame(cols)
to_expand = ['header', 'images']
for col in to_expand:
    temp_df = pd.DataFrame(df[col].tolist(), index=df.index)
    df = df.join(temp_df)

df.drop(columns=['header', 'BAND', 'images'], inplace=True)

# shuffle
df = df.sample(frac=1, random_state=SEED) if SEED else df.sample(frac=1)

In [40]:
# GET SUBTRACTED AND DIVIDED BANDS IF NEEDED

if band_choice != 'norm':

    def calculate_ratios(row, sigma=5):
        for numerator in numers:
            num = row[numerator]
            den = row[denominator]
            ratio = ut.masked_ratio(num, den)
            row[numerator + '/' + denominator] = ut.clip_extremes(ratio, sigma)
        return row

    numers = ['f150w', 'f277w']
    denominator = 'f444w'

    for numerator in numers:
        df[denominator + '-' + numerator] = df[denominator] - df[numerator]

    df = df.apply(calculate_ratios, axis=1)

In [41]:
# STACK AND RESIZE BANDS

resize_transform = transforms.Resize(
    INPUT_SIZE, transforms.InterpolationMode.BILINEAR
)

df['images'] = df[BANDS].values.tolist()
df['images'] = [resize_transform(torch.as_tensor(img)) for img in df['images']]

In [ ]:
# EXTRACT X, y, and i from data

data_df = df[df['TABLE'] != 'test']

X_data = torch.stack(data_df['images'].tolist())
y_data = torch.tensor(data_df['FLAG'].tolist())
i_data = torch.tensor(data_df['idx'].astype(int).tolist())

X_data.shape

In [44]:
# SCALE DATA


def scale_data(data, type='tanh'):

    if type == 'log1p':
        data = torch.log1p(data)
    elif type == 'pytorch_normalize':
        scaler = Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
        data = scaler(data)
    elif type == 'arcsinh':
        data = torch.asinh(data)
    elif type == 'tanh':
        data = torch.tanh(data)
    else:
        raise ValueError(f'{type} is not a valid scaling function')

    return data


X_data_scaled = scale_data(X_data, type=SCALER)

ut.analyze_data(X_data, axis=(0, 2, 3), name='Unscaled data:')
print('\n')
ut.analyze_data(X_data_scaled, axis=(0, 2, 3), name=f'{SCALER} Scaled data:')

Unscaled data:
Mean:  [0.03910705 0.09731541 0.1641901 ]
Median:  [0.01090821 0.01966853 0.02870113]
St. Dev:  [0.15515302 0.3676038  0.74392295]
Max:  [17.483625 34.802776 80.58712 ]
Min:  [-0.3531548  -0.21392332 -0.3174752 ]
Length:  438


tanh Scaled data:
Mean:  [0.03511126 0.07637279 0.10358745]
Median:  [0.01090777 0.01966599 0.02869325]
St. Dev:  [0.08709539 0.15428431 0.20217855]
Max:  [1. 1. 1.]
Min:  [-0.33917043 -0.21071869 -0.3072222 ]
Length:  438


In [ ]:
# CONVERT TO DATSET

# transforms to apply to inputs
transform = transforms.Compose(
    [
        # transforms.RandomHorizontalFlip(p=0.5),
        # transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomRotation(degrees=(0, 360))
        # transforms.RandomPerspective()
    ]
)

dataset = ut.GalaxyDataset(
    images=X_data_scaled,
    labels=y_data,
    transform=transform,
    multiplier=AUG_MULT,
)

## Define Model

In [46]:
# DEFINE MODEL AND HYPERPARAMTER FUNCTIONS


def define_model(trial):
    # get model structure from optuna
    n_conv = trial.suggest_int('n_conv', 1, 3)
    n_fc = trial.suggest_int('n_fc', 1, 3)

    conv_channels = [
        trial.suggest_int(f'n_channels_conv{i}', 1, 32) for i in range(3)
    ]
    conv_ksizes = [
        trial.suggest_int(f'kernel_size_conv{i}', 1, 3) for i in range(3)
    ]
    conv_paddings = [
        trial.suggest_int(f'padding_conv{i}', 1, 3) for i in range(3)
    ]
    conv_dropouts = [
        trial.suggest_float(f'dropout_conv{i}', 0.2, 0.5) for i in range(3)
    ]
    fc_units = [
        trial.suggest_int(f'n_units_fc{i}', 4, 256, step=4) for i in range(3)
    ]
    fc_dropouts = [
        trial.suggest_float(f'dropout_fc{i}', 0.2, 0.5) for i in range(3)
    ]

    layers = []

    # build conv layers
    in_channels = len(BANDS)
    current_spatial_size = INPUT_SIZE

    for i in range(n_conv):
        out_channels = conv_channels[i]
        ksize = conv_ksizes[i]
        pad = conv_paddings[i]
        dropout = conv_dropouts[i]

        layers.extend(
            (
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=ksize,
                    padding=pad,
                    bias=False,
                ),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        current_spatial_size = current_spatial_size - ksize + (2 * pad) + 1
        in_channels = out_channels

    flattened_features = in_channels * (current_spatial_size**2)
    layers.append(nn.Flatten())

    # build fc layers
    in_features = flattened_features
    for i in range(n_fc):
        out_features = fc_units[i]
        dropout = fc_dropouts[i]

        layers.extend(
            (
                nn.Linear(in_features, out_features, bias=False),
                nn.BatchNorm1d(out_features),
                nn.ReLU(),
                nn.Dropout(dropout),
            )
        )

        in_features = out_features

    layers.append(nn.Linear(in_features, 2 * len(OUT_FEATURES)))

    return nn.Sequential(*layers)


def get_hyperparameters(trial, model):

    # optimizer and learning rate
    lr = trial.suggest_float('lr', 1e-5, 1e-1, log=True)
    optimizer_name = trial.suggest_categorical(
        'optimizer', ['Adam', 'SGD', 'AdamW']
    )

    kwargs = {'lr': lr}

    if optimizer_name == 'AdamW':
        kwargs['weight_decay'] = trial.suggest_float(
            'adamw_weight_decay', 1e-5, 1e-1, log=True
        )
    elif optimizer_name == 'SGD':
        kwargs['momentum'] = trial.suggest_float('sgd_momentum', 0.0, 0.99)

    optimizer = getattr(torch.optim, optimizer_name)(
        model.parameters(), **kwargs
    )

    # weights and loss function
    w1 = trial.suggest_float('w1', 1, 3)
    w2 = trial.suggest_float('w2', 1, 3)
    weights = torch.tensor([w1, w2]).to(DEVICE)
    loss_func = nn.CrossEntropyLoss(weight=weights)

    # batch size and loaders
    batch_size = trial.suggest_categorical('batch_size', [16, 32, 64])

    return optimizer, loss_func, batch_size

In [ ]:
# DEFINE OBJECTIVE FUNCTION


def objective(trial):
    global BEST_ACC

    kfold = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_accuracies = []

    # Split the dataset into 5 folds
    for fold, (train_idx, val_idx) in enumerate(kfold.split(dataset)):
        # create new model
        model = define_model(trial).to(DEVICE)
        optimizer, loss_func, batch_size = get_hyperparameters(trial, model)

        # get separate val and train features and labels and scale
        features, labels = dataset.images, dataset.labels
        X_train, X_val = features[train_idx], features[val_idx]
        y_train, y_val = labels[train_idx], labels[val_idx]

        # create subsets and dataloaders
        train_subset = ut.GalaxyDataset(
            images=X_train,
            labels=y_train,
        )
        val_subset = ut.GalaxyDataset(images=X_val, labels=y_val)

        if SEED:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=False,
                drop_last=True,
            )
        else:
            train_loader = torch.utils.data.DataLoader(
                train_subset,
                batch_size=batch_size,
                shuffle=True,
                drop_last=True,
            )
        val_loader = torch.utils.data.DataLoader(
            val_subset, batch_size=batch_size, shuffle=False
        )

        # early stopper
        stopper = ut.EarlyStopper(patience=PATIENCE, min_delta=0.0005)
        for _ in range(EPOCHS):
            # Training Loop ----------------------------------------------------
            model.train()
            ut.do_epoch(
                model=model,
                loader=train_loader,
                loss_function=loss_func,
                device=DEVICE,
                optimizer=optimizer,
                train=True,
            )

            # Validation Loop ---------------------------------------------------
            model.eval()
            with torch.no_grad():
                running_loss, correct = ut.do_epoch(
                    model=model,
                    loader=val_loader,
                    loss_function=loss_func,
                    device=DEVICE,
                    train=False,
                )

                # compute accuracy
                accuracy = correct / len(val_subset)
                if accuracy > BEST_ACC:
                    BEST_ACC = float(accuracy)

                # track loss once per epoch to check early stopper
                val_loss = running_loss / len(val_subset)
                stopper(val_loss)
                if stopper.early_stop:
                    break

        fold_accuracies.append(float(accuracy))
        trial.report(accuracy, step=fold)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(fold_accuracies)

## Train Model

In [48]:
# TRAINS MODEL

if OPTIMIZE:
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    # use gaussian process
    sampler = (
        optuna.samplers.GPSampler(seed=SEED)
        if SEED
        else optuna.samplers.GPSampler()
    )

    study = optuna.create_study(
        direction='maximize',
        sampler=sampler,
        pruner=optuna.pruners.MedianPruner(),
    )
    study.optimize(objective, n_trials=NUM_TRIALS, show_progress_bar=True)

    print('Best trial:')
    trial = study.best_trial
    print(f'  Accuracy: {trial.value}')
    print('  Params: ')
    for key, value in trial.params.items():
        print(f'    {key}: {value}')

  0%|          | 0/20 [00:00<?, ?it/s]


[W 2026-11-19 11:22:38,564] Trial 0 failed with parameters: {'n_conv': 2, 'n_fc': 3, 'n_channels_conv0': 24, 'n_channels_conv1': 20, 'n_channels_conv2': 5, 'kernel_size_conv0': 1, 'kernel_size_conv1': 1, 'kernel_size_conv2': 3, 'padding_conv0': 2, 'padding_conv1': 3, 'padding_conv2': 1, 'dropout_conv0': 0.4909729556485983, 'dropout_conv1': 0.4497327922401265, 'dropout_conv2': 0.26370173320348284, 'n_units_fc0': 48, 'n_units_fc1': 48, 'n_units_fc2': 80, 'dropout_fc0': 0.35742692948967136, 'dropout_fc1': 0.3295835055926347, 'dropout_fc2': 0.2873687420594126, 'lr': 0.0028016351587162596, 'optimizer': 'AdamW', 'adamw_weight_decay': 0.0006672367170464204, 'w1': 2.570351922786027, 'w2': 1.3993475643167195, 'batch_size': 32} because of the following error: TypeError("GalaxyDataset.__init__() got an unexpected keyword argument 'ids'").
Traceback (most recent call last):
  File "c:\Users\harty\miniconda3\envs\pytorch\Lib\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    valu

TypeError: GalaxyDataset.__init__() got an unexpected keyword argument 'ids'

In [ ]:
# LOG OPTIMIZATION

if OPTIMIZE:
    logged_params = trial.params.copy()
    logged_params['comment'] = COMMENT
    logged_params['bands'] = BANDS
    logged_params['out_features'] = OUT_FEATURES
    logged_params['seed'] = SEED
    logged_params['epochs'] = EPOCHS
    logged_params['image_size'] = INPUT_SIZE
    logged_params['num_trials'] = NUM_TRIALS
    logged_params['patience'] = PATIENCE
    logged_params['accuracy'] = trial.value
    logged_params['best accuracy'] = BEST_ACC
    logged_params['scaling'] = SCALER
    logged_params['N_folds'] = N_SPLITS
    logged_params['aug_mult'] = AUG_MULT

    logger = ut.RunLogger(filepath=home_dir / 'logs' / 'conv.json')
    logger.log_run(logged_params)